In [1]:
!pip install -q numpy librosa soundfile pandas tqdm
!apt-get install -y -q libsndfile1

Reading package lists...
Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.


In [2]:
import os
import sys
import pickle
import warnings
import librosa
import kagglehub
import numpy as np
import soundfile as sf
import pandas as pd
from collections import Counter
from tqdm import tqdm

In [3]:
utils_path = kagglehub.dataset_download("shafayatulislam/ai-audio-monitor-utils")
print(utils_path)

/kaggle/input/datasets/shafayatulislam/ai-audio-monitor-utils


In [4]:
sys.path.insert(0, "/kaggle/input/datasets/shafayatulislam/ai-audio-monitor-utils")

In [5]:
from audio_utils import (
    TARGET_SR,
    CHUNK_SAMPLES,
    preprocess_signal,
    extract_features,
)

In [6]:
print(f"TARGET_SR = {TARGET_SR} Hz")
print(f"Nyquist ceiling = {TARGET_SR // 2} Hz")
print(f"CHUNK_SAMPLES = {CHUNK_SAMPLES} for {CHUNK_SAMPLES / TARGET_SR:.1f} s")

TARGET_SR = 16000 Hz
Nyquist ceiling = 8000 Hz
CHUNK_SAMPLES = 16000 for 1.0 s


## Load Datasets

In [7]:
data_path = kagglehub.dataset_download("shafayatulislam/ai-audio-monitor-datasets")
print(data_path)

/kaggle/input/datasets/shafayatulislam/ai-audio-monitor-datasets


In [8]:
# Initialize paths
cough_audio = ""
noise_audio = ""
noise_meta = ""
repo_audio = ""

In [9]:
for current_dir, dirs, files in os.walk(data_path):
    if current_dir.endswith("cough-detection/audio"):
        cough_audio = current_dir
    if current_dir.endswith("noise-robustness/audio"):
        noise_audio = current_dir
    if current_dir.endswith("respiratory-conditions/audio_and_txt_files"):
        repo_audio = current_dir
    if "esc50.csv" in files:
        noise_meta = os.path.join(current_dir, "esc50.csv")

In [10]:
print(f"cough_audio: {cough_audio}")
print(f"noise_audio: {noise_audio}")
print(f"noise_meta: {noise_meta}")
print(f"repo_audio: {repo_audio}")

cough_audio: /kaggle/input/datasets/shafayatulislam/ai-audio-monitor-datasets/cough-detection/cough-detection/audio
noise_audio: /kaggle/input/datasets/shafayatulislam/ai-audio-monitor-datasets/noise-robustness/noise-robustness/audio
noise_meta: /kaggle/input/datasets/shafayatulislam/ai-audio-monitor-datasets/noise-robustness/noise-robustness/meta/esc50.csv
repo_audio: /kaggle/input/datasets/shafayatulislam/ai-audio-monitor-datasets/respiratory-conditions/respiratory-conditions/audio_and_txt_files


In [11]:
MAX_SAMPLES_PER_CLASS = 3000

In [12]:
def make_chunk_feature(signal: np.ndarray, start_sample: int):
    chunk = signal[start_sample : start_sample + CHUNK_SAMPLES]
    if len(chunk) < CHUNK_SAMPLES:
        return None
    return extract_features(chunk)

In [13]:
eda = []
count = {0: 0, 1: 0, 2: 0}

In [14]:
def load_audio(filepath: str):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            signal, _ = librosa.load(filepath, sr=TARGET_SR)
            return signal
        except Exception as e:
            print(f"Skipping corrupted/unreadable file: {os.path.basename(filepath)}")
            return None

## Respiratory Condition

In [15]:
def parse_icbhi(txt_path: str):
    segments = []
    with open(txt_path) as f:
        for line in f:
            try:
                s, e, c, w = line.strip().split()
                label = 2 if (int(c) or int(w)) else 0
                segments.append((float(s), float(e), label))
            except Exception:
                continue
    return segments

In [16]:
for file in tqdm(sorted(os.listdir(repo_audio)), desc="D1"):
    if not file.endswith(".wav"):
        continue
    wav = os.path.join(repo_audio, file)
    txt = wav.replace(".wav", ".txt")
    if not os.path.exists(txt):
        continue

    signal = load_audio(wav) 
    if signal is None:
        continue

    for start_sec, end_sec, label in parse_icbhi(txt):
        if count[label] >= MAX_SAMPLES_PER_CLASS:
            continue
        feat = make_chunk_feature(signal, int(start_sec * TARGET_SR))
        if feat is not None:
            eda.append((feat, label))
            count[label] += 1

print(count)

D1: 100%|██████████| 1840/1840 [01:43<00:00, 17.76it/s]

{0: 3000, 1: 0, 2: 3000}


## Cough Detection

In [17]:
def augment_and_add(signal, label, augs_needed):
    global count, eda
    aug_count = 0
    
    feat = make_chunk_feature(signal, 0)
    if feat is not None:
        eda.append((feat, label))
        count[label] += 1
        
    for noise_factor in [0.002, 0.004, 0.006, 0.008, 0.01]:
        if count[label] >= MAX_SAMPLES_PER_CLASS or aug_count >= augs_needed: return
        noisy_sig = signal + noise_factor * np.random.randn(len(signal))
        feat = make_chunk_feature(noisy_sig, 0)
        if feat is not None: eda.append((feat, label)); count[label] += 1; aug_count += 1

    for rate in [0.8, 0.85, 0.9, 1.1, 1.15, 1.2]:
        if count[label] >= MAX_SAMPLES_PER_CLASS or aug_count >= augs_needed: return
        stretched = librosa.effects.time_stretch(signal, rate=rate)
        feat = make_chunk_feature(stretched, 0)
        if feat is not None: eda.append((feat, label)); count[label] += 1; aug_count += 1
            
    for steps in [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]:
        if count[label] >= MAX_SAMPLES_PER_CLASS or aug_count >= augs_needed: return
        pitched = librosa.effects.pitch_shift(signal, sr=TARGET_SR, n_steps=steps)
        feat = make_chunk_feature(pitched, 0)
        if feat is not None: eda.append((feat, label)); count[label] += 1; aug_count += 1

In [18]:
for file in tqdm(sorted(os.listdir(cough_audio)), desc="D2"):
    if not file.endswith(".wav"):
        continue
    name  = file.lower()
    label = 1 if "cough" in name else (0 if "breathing" in name else None)
    
    if label is None or count[label] >= MAX_SAMPLES_PER_CLASS:
        continue

    signal = load_audio(os.path.join(cough_audio, file))
    if signal is None:
        continue

    if label == 1:
        augment_and_add(signal, label, augs_needed=24)
    else:
        feat = make_chunk_feature(signal, 0)
        if feat is not None:
            eda.append((feat, label))
            count[label] += 1

print(count)

D2:  41%|████      | 70/170 [00:22<00:25,  3.91it/s]

Skipping corrupted/unreadable file: 20200820_._cough-heavy.wav


D2:  91%|█████████ | 154/170 [00:43<00:05,  2.80it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(
D2: 100%|██████████| 170/170 [00:47<00:00,  3.57it/s]

{0: 3000, 1: 1494, 2: 3000}


## Noise Robustness

In [19]:
meta = pd.read_csv(noise_meta)

for i in tqdm(range(len(meta)), desc="D3"):
    row = meta.iloc[i]
    
    label = 1 if row["category"] == "coughing" else 0
    
    if count[label] >= MAX_SAMPLES_PER_CLASS:
        continue

    filepath = os.path.join(noise_audio, row["filename"])
    if not os.path.exists(filepath):
        continue

    signal = load_audio(filepath)
    if signal is None:
        continue

    if label == 1:
        augment_and_add(signal, label, augs_needed=24)
    else:
        feat = make_chunk_feature(signal, 0)
        if feat is not None:
            eda.append((feat, label))
            count[label] += 1

print(count)

D3: 100%|██████████| 2000/2000 [00:19<00:00, 101.41it/s]

{0: 3000, 1: 2214, 2: 3000}


## Save

In [20]:
with open("eda.pkl", "wb") as f:
    pickle.dump(eda, f)
print("eda.pkl saved")

eda.pkl saved


## Summary

In [21]:
print(f"Total samples: {len(eda)}")
print(f"Class distribution: {Counter(y for _, y in eda)}")
print(f"Feature shape: {eda[0][0].shape}")
print(f"Capture SR: {TARGET_SR} Hz")

Total samples: 8214
Class distribution: Counter({0: 3000, 2: 3000, 1: 2214})
Feature shape: (60, 32)
Capture SR: 16000 Hz
